<a href="https://colab.research.google.com/github/ncsu-geoforall-lab/GIS582-assignments/blob/main/6%20-%20Viewshed%2C%20solar%20energy%20potenial%20analysis/6A_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 6: Viewshed, solar energy potential analysis

**Course:** [GIS 582 - Geospatial Modeling and Analysis](https://ncsu-geoforall-lab.github.io/geospatial-modeling-course/grass/viewshed_solar.html)  
**Institution:** [NC State University, Center for Geospatial Analytics](https://cnr.ncsu.edu/geospatial/)
**Instructors:** Helena Mitasova, Corey White, and team

## Learning Objectives

In this tutorial, you will learn how to:
* line of sight, viewshed and cumulative viewshed: principle and applications
* solar radiation: components and dynamics
* solar radiation in complex terrain, cast shadows
* cumulative solar irradiation, solar energy potential

## Tutorial Outline

* Part 1: Environment Setup
* Part 2: Viewshed analysis
* Part 3: Solar irradiation analysis
* Part 4: Optional: Calculate direct solar irradiation and insolation time for a larger region.

---
## Part 1: Environment Setup

### Install GRASS

**Important:** This setup takes 3-5 minutes. You'll need to run it each time you start a new Colab session.

In [ ]:
!add-apt-repository -y ppa:ubuntugis/ubuntugis-unstable
!apt update
!apt-get install -y grass-core grass-dev pdal

Check that GRASS and PDAL are installed by asking which versions are available.

In [ ]:
!grass --version
!pdal --version

Check which Python version is running.

In [ ]:
import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import GRASS Python packages

We need to locate GRASS Python packages using `grass --config python_path` and add them to `sys.path`.

In [ ]:
import subprocess
import os
from pathlib import Path

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj

import csv
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

### Download North Carolina Sample Dataset

This dataset includes elevation, orthophotos, and sample lidar data used in this tutorial.

In [ ]:
!grass --tmp-project XY --exec g.download.project url=https://grass.osgeo.org/sampledata/north_carolina/nc_basic_spm_grass7.tar.gz path=/content

Download the LiDAR and orthophoto data for the area of interest. The data is in LAS format, which is a common format for storing LiDAR point cloud data.

Use the cell below to download the data directly into your Colab environment or access it through the links provided.

* [Folder with LAS file and orthophoto](https://drive.google.com/drive/folders/1C_vov2M85dOmHp7dzLLyW8JP--IrqonY?usp=drive_link)

Alternative links:

* [LAS tile 0793_016 in NC State Plane Meters](http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/tile_0793_016_spm.las)
* [Orthophoto geotif (mosaic at 1ft resolution)](http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/ortho_0793_016_ncspm.zip)

In [ ]:
!curl -L -o tile_0793_016_spm.las http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/tile_0793_016_spm.las
!curl -L -o ortho_0793_016_ncspm.zip http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data/ortho_0793_016_ncspm.zip

### Initialize GRASS Session

In [ ]:
# Start GRASS session
grassdata = "/content"
project = "nc_basic_spm_grass7"
mapset = "user1"  # Create a new mapset for our work

# Start GRASS Session
session = gj.init(Path(project, mapset))
print("GRASS session started.")

In [ ]:
# Set computational region to elevation raster
gs.run_command("g.region", raster="elevation", flags="pg")

---
## Part 2: Viewshed analysis

 Compute viewshed from a 32 story tower located in downtown.

 Write a new file called `viewshed_points.txt` using the **cell magic command `%%writefile` that contians the locations we will use for the viewshed analysis.

In [ ]:
%%writefile viewshed_points.txt
642212,224767,165
638898,224528,25

In [ ]:
!g.region raster=elev_ned_30m -ap
!v.in.ascii -z input=viewshed_points.txt output=viewpoints separator=, z=3
!r.viewshed elev_ned_30m output=tower_165_los coordinates=642212,224767 observer_elevation=165 max_distance=10000

 Display result on shaded relief:

In [ ]:
tower_view_map = gj.Map(filename="towerview_angle.png", height=450, width=450)
tower_view_map.d_his(hue="tower_165_los", intensity="elevation_shade")
tower_view_map.d_vect(map="viewpoints", size=10, color="orange", icon="basic/marker")
tower_view_map.d_barscale(at=[3, 6])
tower_view_map.d_legend(raster="tower_165_los", at=(50,90,2,6))
tower_view_map.show()

 Find out what is the landuse within the view using map algebra: 

In [ ]:
!r.mapcalc "tower_los_lu30m = if(tower_165_los,landclass96,null())"
!r.colors tower_los_lu30m raster=landclass96
!r.category tower_los_lu30m raster=landclass96
!r.report -n tower_los_lu30m unit=p,h

 Display only the following layers and save result: 

In [ ]:
tower_los_lu30m_m = gj.Map(filename="towerviewlu.png", height=450, width=450)
tower_los_lu30m_m.d_rast(map="tower_los_lu30m")
tower_los_lu30m_m.d_legend(raster="tower_los_lu30m", at=(25,50,1,3), flags="cb")
tower_los_lu30m_m.show()

 We can also map visibility from former RedHat headquarters on Centennial Campus:

In [ ]:
!r.viewshed elev_ned_30m output=redhat_25_los coordinates=638898,224528 observer_elevation=25 max_distance=10000

 Display only the following layers and save result:

In [ ]:
redhat_25_los_m = gj.Map(filename="RHview.png", height=450, width=450)
redhat_25_los_m.d_rast(map="redhat_25_los")
redhat_25_los_m.d_vect(map="streets_wake")
redhat_25_los_m.d_vect(map="viewpoints", size=10, col="red", icon="basic/marker")
redhat_25_los_m.show()

#### Question 2.1

 Use map algebra to compute landuse in within the viewshed, assign the visible land use map land use colors and category labels using [r.colors](https://grass.osgeo.org/grass-stable/manuals/r.colors.html) and [r.category](https://grass.osgeo.org/grass-stable/manuals/r.category.html) (see previous example) and use [r.report -n](https://grass.osgeo.org/grass-stable/manuals/r.report.html) to **compare the size and land use composition within viewshed from the downtown tower and the building on Cenetennial Campus that served as RedHat headquarters**. 

`Add text here with your answer to question 2.1`

---
## Part 3: Solar irradiation analysis

Set the region and add the planned building to the DEM, we will use this new DEM for the analyses.

In [ ]:
!g.region rural_1m -p
!r.mapcalc "elevfacility_1m = if(isnull(facility), elev_lid792_1m,138.)"
!r.colors elevfacility_1m color=elevation

Zoom to the region. 

In [ ]:
m = gj.Map(height=450, width=450, use_region=True)
m.d_rast(map="elevfacility_1m")
m.show()

 Prepare input maps (slope and aspect): 

In [ ]:
!r.slope.aspect elevation=elevfacility_1m aspect=aspect_elevfac_1m slope=slope_elevfac_1m

### Incidence angles and cast shadows

 Compute the sun position on Dec. 22 at 2:25pm, EST (no map output expected): 

In [ ]:
!r.sunmask -s elevation=elevfacility_1m year=2001 month=12 day=22 hour=14 minute=25 sec=0 timezone=-5

Calculate incidence angles including cast shadows on Dec. 22 at 2:25pm, EST

In [ ]:
!r.sun elevation=elevfacility_1m aspect=aspect_elevfac_1m slope=slope_elevfac_1m incidout=incid_elevfac_1m day=356 time=14.416667
!r.info incid_elevfac_1m
!r.colors -e incid_elevfac_1m co=bcyr

Display the map:

In [ ]:
m = gj.Map(filename="incid_angle.png", height=450, width=450, use_region=True)
m.d_rast(map="incid_elevfac_1m")
m.d_legend(raster="incid_elevfac_1m", at=(25,50,1,3))
m.show()

#### Question 3.1

We assign histogram equalized color table - can you explain why?
What is the value of the incidence angle on the roof? How is it related to solar altitude? (hint: see the output of r.info and query the incidence angle map on the building)

`Add text here with your answer to question 3.1`

 Extract the cast shadow area for 14.4 hr.

In [ ]:
!r.mapcalc "shadow_1m = if(isnull(incid_elevfac_1m), 1, null())"
!r.colors shadow_1m color=grey

In [ ]:
m = gj.Map(filename="shadows14hr.png", height=450, width=450, use_region=True)
m.d_rast(map="elevfacility_1m")
m.d_rast(map="shadow_1m")
m.show()

and then compute an incidence angle map and extract shadow area for the morning at 7.5 hr same day:

In [ ]:
!r.sun elevation=elevfacility_1m aspect=aspect_elevfac_1m slope=slope_elevfac_1m incidout=incid_elevfac7_1m day=356 time=7.50
!r.mapcalc "shadow7_1m = if(isnull(incid_elevfac7_1m), 1, null())"
!r.colors shadow7_1m color=grey

In [ ]:
m = gj.Map(filename="shadows7hr.png", height=450, width=450, use_region=True)
m.d_rast(map="elevfacility_1m")
m.d_rast(map="shadow7_1m")
m.show()

#### Question 3.2

 How does the cast shadow area change between the morning and early afternoon on the day of winter solstice? (hint, use r.report) Is there any overlap?

`Add text here with your answer to question 3.2`

### Solar irradiation

Compute global (beam+diffuse+refl) irradiation for entire day during summer and winter solstice.
Display the irradiation maps and insolation time maps with legends to compare the range of values

Compute the irradiance/irradiation and insolation time of the solar winter (day=365)

In [ ]:
!r.sun elevation=elevfacility_1m aspect=aspect_elevfac_1m slope=slope_elevfac_1m day=356 glob_rad=g356 insol_time=its356
!r.colors g356 co=gyr -e

Display the irradiance/irradiation [Wh.m-2.day-1]

In [ ]:
m = gj.Map(filename="solar_winter.png", height=450, width=450, use_region=True)
m.d_rast(map="g356")
m.d_legend(raster="g356", at=(25,50,1,3))
m.show()

Now display the insolation time

In [ ]:
m = gj.Map(filename="solartime_winter.png", height=450, width=450, use_region=True)
m.d_rast(map="its356")
m.d_legend(raster="its356", at=(25,50,1,3))
m.show()

Compute irradiance/irradiation and insolation time of the solar summer (day=172)

In [ ]:
!r.sun elevation=elevfacility_1m aspect=aspect_elevfac_1m slope=slope_elevfac_1m day=172 glob_rad=g172 insol_time=its172
!r.colors g172 co=gyr -e

In [ ]:
m = gj.Map(filename="solartime_summer.png", height=450, width=450, use_region=True)
m.d_rast(map="its172")
m.d_legend(raster="its172", at=(25,50,1,3))
m.show()

#### Question 3.3

Discuss the difference between the daily solar irradiation values and insolation time during the days of summer and winter solstice. What are the units for solar irradiation? What are the implications for using solar irradiation for generating sufficient energy throughout the year?

Optionally display the irradiation maps draped over elevation elevfacility_1m in 3D view to see relation between terrain geometry and solar irradiation. 

---
## Part 4: Optional: Calculate direct solar irradiation and insolation time for a larger region.

 Try to find good color tables (custom, hist. equalized, to see the pattern).

In [ ]:
!g.region raster=elev_ned_30m -p
!r.slope.aspect elevation=elev_ned_30m aspect=asp_ned_30m slope=slp_ned_30m
!r.sun elevation=elev_ned_30m aspect=asp_ned_30m slope=slp_ned_30m linke_value=2.5 albedo_value=0.2 beam_rad=b356 diff_rad=d356 refl_rad=r356 insol_time=it356 day=356
!r.colors b356,d356,r356 co=bcyr -e

Visualzie the following layers:

| layer | filename |
|-------|----------|
| b365  | beamrad356.png |
| it356 | insol_time.png |


 Dowload grass addon script to compute daily solar irradiation for 10 days, save the time series output as a space-time dataset and visualize the result using animation.

In [ ]:
!g.extension r.sun.daily
!g.region raster=elevation -p
!r.sun.daily elevation=elevation start_day=30 end_day=40 beam_rad_basename=beam beam_rad=beam_sum nprocs=4 -t
!t.info beam
!r.info beam

Create an animation from the time series output.

In [ ]:
mimg = TimeSeriesMap("beam")
mimg.d_legend()  # Add legend
mimg.show()  # Create TimeSlider
mimg.save("beam.gif")